In [ ]:

!pip -q install nilearn nibabel scikit-learn

import os, gc, copy, json, math, random, time, warnings
import numpy as np
import nibabel as nib
from nilearn import datasets

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models.video as models_video
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             average_precision_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

# --- (опционально) Google Drive для устойчивости к сбросу среды ---
USE_DRIVE = True
try:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = "/content/drive/MyDrive/aes_results"
    else:
        RESULTS_DIR = "/content/aes_results"
except Exception as e:
    print("Drive не примонтирован, пишу локально:", e)
    RESULTS_DIR = "/content/aes_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("RESULTS_DIR:", RESULTS_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 155.7 MB/s eta 0:00:00
Device: cuda | GPU: NVIDIA A100-SXM4-80GB
Mounted at /content/drive
RESULTS_DIR: /content/drive/MyDrive/aes_results


In [ ]:
# ============================================================
# CONFIG (v3)
# ============================================================
from dataclasses import dataclass

@dataclass
class CFG:
    seed: int = 42
    data_dir: str = "/content/abide_data"
    cache_dir: str = "/content/abide_cache"     # локально: распакованные .pt (быстро пересоздаётся)
    n_subjects: int = 150
    pipeline: str = "cpac"
    derivative: str = "func_preproc"

    seq_length: int = 30
    batch_size: int = 8        # A100/H100 -> 8 (было 2)
    num_workers: int = 4
    pin_memory: bool = True
    use_amp: bool = True       # mixed precision

    epochs: int = 15
    epochs_ablation: int = 15
    epochs_cv: int = 15
    lr: float = 1e-4
    weight_decay: float = 1e-3
    grad_clip: float = 1.0

    max_alpha: float = 0.05
    compression_ratio: float = 0.25
    warmup_epochs: int = 4
    deep_layer: str = "layer4"
    early_layer: str = "layer1"

    l2_weight_decay: float = 1e-2
    l1_act_lambda: float = 1e-4
    vib_latent: int = 128
    vib_beta_max: float = 1e-3

    val_size: float = 0.2
    n_folds: int = 5

CFG = CFG()


In [ ]:
# ============================================================
# DATA + ОДНОРАЗОВЫЙ КЭШ (ключевое ускорение)
#   .nii.gz распаковывается ОДИН раз -> /content/abide_cache/{idx}.pt (float16, raw)
#   z-score применяется уже при обучении (как в исходнике), на окне.
# ============================================================
seed_everything(CFG.seed)
print("Downloading ABIDE...")
abide = datasets.fetch_abide_pcp(data_dir=CFG.data_dir, n_subjects=CFG.n_subjects,
                                 pipeline=CFG.pipeline, derivatives=[CFG.derivative])
fmri_files = np.array(list(abide.func_preproc), dtype=object)
labels = np.array([1 if dx == 1 else 0 for dx in abide.phenotypic["DX_GROUP"]], dtype=int)
print(f"Files: {len(fmri_files)} | ASD: {labels.sum()} | Control: {(labels==0).sum()}")

os.makedirs(CFG.cache_dir, exist_ok=True)
def build_cache():
    t0 = time.time()
    for i, f in enumerate(fmri_files):
        p = f"{CFG.cache_dir}/{i}.pt"
        if os.path.exists(p):
            continue
        d = nib.load(f).get_fdata()                                  # [H,W,D,T]
        x = torch.tensor(d, dtype=torch.float32).permute(3,0,1,2)    # [T,H,W,D]
        torch.save(x.to(torch.float16), p)                           # raw, без z-score
        if (i+1) % 25 == 0:
            print(f"  cached {i+1}/{len(fmri_files)}  ({time.time()-t0:.0f}s)")
    print(f"Cache ready in {time.time()-t0:.0f}s  ->  {CFG.cache_dir}")
build_cache()

all_idx = np.arange(len(fmri_files))
# канонический сплит (для секций D и E)
train_idx, val_idx = train_test_split(all_idx, test_size=CFG.val_size,
                                       random_state=CFG.seed, stratify=labels)
print(f"Single split -> Train: {len(train_idx)} | Val: {len(val_idx)}")


[fetch_abide_pcp] Added README.md to /content/abide_data

[fetch_abide_pcp] Dataset created in /content/abide_data/ABIDE_pcp

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv ...

[fetch_abide_pcp]  ...done. (2 seconds, 0 min)

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/func_preproc/Pitt_005
0003_func_preproc.nii.gz ...

...

[fetch_abide_pcp] Downloaded 49152000 of 86334444 bytes (56.9%%,    4.0s remaining)

[fetch_abide_pcp] Downloaded 63471616 of 86334444 bytes (73.5%%,    2.3s remaining)

[fetch_abide_pcp] Downloaded 77889536 of 86334444 bytes (90.2%%,    0.8s remaining)

[fetch_abide_pcp]  ...done. (9 seconds, 0 min)

Files: 150 | ASD: 77 | Control: 73
  cached 25/150  (56s)
  cached 50/150  (111s)
  cached 75/150  (170s)
  cached 100/150  (197s)
  cached 125/150  (244s)
  cached 150/150  (288s)
Cache ready in 288s  ->  /content/abide_cache
Single split -> Train: 120 | Val: 30


In [ ]:
# ============================================================
# DATASET — читает из кэша .pt (быстро; OS page cache держит файлы в RAM между эпохами)
# ============================================================
class ABIDECached(Dataset):
    def __init__(self, idxs, labels_all, seq_length=30, mode="train"):
        self.idxs = list(idxs); self.labels_all = labels_all
        self.seq_length = seq_length; self.mode = mode
    def __len__(self): return len(self.idxs)
    def _crop(self, x):
        T = x.shape[0]
        if T >= self.seq_length:
            s = random.randint(0, T-self.seq_length) if self.mode=="train" else (T-self.seq_length)//2
            x = x[s:s+self.seq_length]
        else:
            x = torch.cat([x, torch.zeros((self.seq_length-T, *x.shape[1:]), dtype=x.dtype)], 0)
        return x
    def __getitem__(self, k):
        gi = self.idxs[k]
        x = torch.load(f"{CFG.cache_dir}/{gi}.pt", map_location="cpu").float()  # [T,H,W,D]
        x = self._crop(x)
        x = (x - x.mean()) / (x.std() + 1e-8)     # z-score окна (как в исходнике)
        x = x.unsqueeze(1)                         # [T,1,H,W,D]
        y = torch.tensor(int(self.labels_all[gi]), dtype=torch.long)
        return x, y

def make_loaders(tr_idx, va_idx, cfg=CFG):
    tr = ABIDECached(tr_idx, labels, cfg.seq_length, "train")
    va = ABIDECached(va_idx, labels, cfg.seq_length, "val")
    tl = DataLoader(tr, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.num_workers, pin_memory=cfg.pin_memory, drop_last=False)
    vl = DataLoader(va, batch_size=cfg.batch_size, shuffle=False,
                    num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    return tl, vl


In [ ]:
# ============================================================
# MODEL (3D-ResNet18 + LSTM, опциональный VIB) — kl в float для стабильности под AMP
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, lstm_hidden=256, num_classes=2, lstm_dropout=0.3,
                 clf_dropout=0.4, vib=False, vib_latent=128):
        super().__init__()
        self.encoder = models_video.r3d_18(weights=None)
        st = self.encoder.stem[0]
        self.encoder.stem[0] = nn.Conv3d(1, st.out_channels, kernel_size=st.kernel_size,
                                         stride=st.stride, padding=st.padding, bias=False)
        self.encoder.fc = nn.Identity()
        self.vib = vib
        if vib:
            self.to_stats = nn.Linear(512, 2*vib_latent); lstm_in = vib_latent
        else:
            lstm_in = 512
        self.lstm = nn.LSTM(lstm_in, lstm_hidden, num_layers=2, batch_first=True, dropout=lstm_dropout)
        self.classifier = nn.Sequential(nn.Linear(lstm_hidden,128), nn.ReLU(),
                                        nn.Dropout(clf_dropout), nn.Linear(128,num_classes))
    def forward(self, x):
        B,T,C,H,W,D = x.shape
        f = self.encoder(x.view(B*T,C,H,W,D))               # [B*T,512]
        kl = torch.zeros((), device=x.device)
        if self.vib:
            mu, logvar = self.to_stats(f).chunk(2, dim=-1)
            if self.training:
                f = mu + torch.exp(0.5*logvar)*torch.randn_like(logvar)
            else:
                f = mu
            mu32, lv32 = mu.float(), logvar.float()
            kl = (-0.5*(1 + lv32 - mu32.pow(2) - lv32.exp())).sum(-1).mean()
        _, (h, _) = self.lstm(f.view(B,T,-1))
        return self.classifier(h[-1]), kl


In [ ]:
# ============================================================
# AES LOSS — энтропия считается в float (под AMP)
# ============================================================
class AdaptiveDynamicIBLoss(nn.Module):
    def __init__(self, max_alpha=0.05, compression_ratio=0.25):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(); self.max_alpha=max_alpha; self.cr=compression_ratio
    @staticmethod
    def _H(act):
        s = F.softmax(act.float(), dim=-1)
        return -(s*torch.log2(s+1e-8)).sum(-1).mean()
    def forward(self, logits, y, early_acts, deep_acts, epoch, warmup=4):
        base = self.ce(logits, y)
        with torch.no_grad():
            target_H = self._H(early_acts[0]) * self.cr
        alpha = self.max_alpha*(epoch+1)/warmup if epoch < warmup else self.max_alpha
        pen = sum(torch.relu(self._H(a) - target_H) for a in deep_acts)
        return base + alpha*pen


In [ ]:
# ============================================================
# МЕТРИКИ
# ============================================================
def full_metrics(t, p, yhat):
    t=np.asarray(t); p=np.asarray(p); yhat=np.asarray(yhat)
    tn,fp,fn,tp = confusion_matrix(t,yhat,labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else float("nan")
    try: auc=roc_auc_score(t,p)
    except ValueError: auc=float("nan")
    try: ap=average_precision_score(t,p)
    except ValueError: ap=float("nan")
    return {"accuracy":accuracy_score(t,yhat), "bal_accuracy":balanced_accuracy_score(t,yhat),
            "precision":precision_score(t,yhat,zero_division=0),
            "recall_sens":recall_score(t,yhat,zero_division=0), "specificity":spec,
            "f1":f1_score(t,yhat,zero_division=0), "roc_auc":auc, "pr_auc":ap,
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval(); T,P,Y=[],[],[]
    for x,y in loader:
        x=x.to(device, non_blocking=True)
        with autocast(enabled=CFG.use_amp):
            logits,_=model(x)
        P.extend(torch.softmax(logits.float(),1)[:,1].cpu().tolist())
        Y.extend(torch.argmax(logits,1).cpu().tolist()); T.extend(y.tolist())
    return full_metrics(T,P,Y)

@torch.no_grad()
def compute_hidden_entropy(model, loader, layer="layer4"):
    model.eval(); acts=[]
    h=getattr(model.encoder,layer).register_forward_hook(lambda m,i,o: acts.append(o.view(o.shape[0],-1)))
    ent=[]
    for x,_ in loader:
        x=x.to(device, non_blocking=True); acts.clear()
        with autocast(enabled=CFG.use_amp):
            _=model(x)
        if acts:
            s=F.softmax(acts[0].float(),dim=-1)
            ent.append(float(-(s*torch.log2(s+1e-8)).sum(-1).mean()))
    h.remove()
    return float(np.mean(ent)) if ent else float("nan")


In [ ]:
# ============================================================
# ОБУЧЕНИЕ (AMP + GradScaler).  method in {baseline,l2,dropout,l1_act,vib,aes}
# ============================================================
def train_one(method, tl, vl, cfg=CFG, epochs=None, overrides=None, verbose=True, tag=""):
    o = dict(gamma=cfg.compression_ratio, alpha=cfg.max_alpha, warmup=cfg.warmup_epochs,
             deep_layer=cfg.deep_layer, early_layer=cfg.early_layer,
             l1_lambda=cfg.l1_act_lambda, vib_beta=cfg.vib_beta_max)
    if overrides: o.update(overrides)
    epochs = epochs or cfg.epochs
    seed_everything(cfg.seed)

    vib = (method=="vib")
    model = HybridModel(lstm_dropout=0.5 if method=="dropout" else 0.3,
                        clf_dropout=0.6 if method=="dropout" else 0.4,
                        vib=vib, vib_latent=cfg.vib_latent).to(device)
    wd = cfg.l2_weight_decay if method=="l2" else cfg.weight_decay
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=wd)
    scaler = GradScaler(enabled=cfg.use_amp)

    early, deep, handles = [], [], []
    if method in ("aes","l1_act"):
        handles.append(getattr(model.encoder,o["early_layer"]).register_forward_hook(
            lambda m,i,out: early.append(out.view(out.shape[0],-1))))
        handles.append(getattr(model.encoder,o["deep_layer"]).register_forward_hook(
            lambda m,i,out: deep.append(out.view(out.shape[0],-1))))
    aes = AdaptiveDynamicIBLoss(o["alpha"], o["gamma"])

    best_state, best_m, best_key = None, None, (-1.0,-1.0)
    for epoch in range(epochs):
        model.train(); run=0.0
        for x,y in tl:
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if method in ("aes","l1_act"): early.clear(); deep.clear()
            with autocast(enabled=cfg.use_amp):
                logits,kl = model(x)
                if method=="aes":
                    loss = aes(logits,y,early,deep,epoch,o["warmup"])
                elif method=="vib":
                    beta=o["vib_beta"]*min(1.0,(epoch+1)/max(1,cfg.warmup_epochs))
                    loss=F.cross_entropy(logits,y)+beta*kl
                elif method=="l1_act":
                    pen=sum(a.float().abs().mean() for a in deep) if deep else 0.0
                    loss=F.cross_entropy(logits,y)+o["l1_lambda"]*pen
                else:
                    loss=F.cross_entropy(logits,y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer); scaler.update(); run+=float(loss.item())
        vm = evaluate_model(model, vl)
        key=(vm["accuracy"], 0.0 if np.isnan(vm["roc_auc"]) else vm["roc_auc"])
        if key>best_key: best_key=key; best_m=vm; best_state=copy.deepcopy(model.state_dict())
        if verbose:
            print(f"[{tag or method}] ep{epoch+1:02d}/{epochs} loss={run/len(tl):.4f} "
                  f"acc={vm['accuracy']:.3f} auc={vm['roc_auc']:.3f} f1={vm['f1']:.3f}")
    for h in handles: h.remove()
    model.load_state_dict(best_state)
    best_m["hidden_entropy_bits"]=compute_hidden_entropy(model, vl, o["deep_layer"])
    del optimizer, model; gc.collect(); torch.cuda.empty_cache()
    return best_m


In [ ]:
# ============================================================
# ПЕРСИСТЕНТНОСТЬ РЕЗУЛЬТАТОВ  (resume после сброса среды)
# ============================================================
import pandas as pd
RESULTS_PATH = f"{RESULTS_DIR}/results.json"
def load_R():
    if os.path.exists(RESULTS_PATH):
        return json.load(open(RESULTS_PATH))
    return {"compare":{}, "ablation":{}, "cv":{}}
def save_R(R):
    json.dump(R, open(RESULTS_PATH,"w"), ensure_ascii=False, indent=1)
def clean(m):  # выкинуть тяжёлые поля, оставить числа
    return {k:v for k,v in m.items() if isinstance(v,(int,float))}

COMPARE_METHODS = ["baseline","l2","dropout","l1_act","vib","aes"]
NAMES_RU = {"baseline":"Базовая (CE)","l2":"L2 (weight decay)","dropout":"Dropout",
            "l1_act":"L1 активаций","vib":"VIB (стохастич. IB)","aes":"AES (предложенный)"}
print("Уже посчитано:", {k:list(v.keys()) for k,v in load_R().items()})


Уже посчитано: {'compare': ['baseline', 'l2', 'dropout', 'l1_act', 'vib', 'aes'], 'ablation': ['gamma=0.1', 'gamma=0.25', 'gamma=0.5', 'gamma=0.75', 'alpha=0.0', 'alpha=0.01', 'alpha=0.05', 'alpha=0.1', 'deep=layer3', 'deep=layer4', 'warmup=0', 'warmup=4'], 'cv': ['fold1:baseline', 'fold1:vib', 'fold1:aes', 'fold2:baseline', 'fold2:vib', 'fold2:aes', 'fold3:baseline', 'fold3:vib', 'fold3:aes', 'fold4:baseline', 'fold4:vib', 'fold4:aes', 'fold5:baseline', 'fold5:vib', 'fold5:aes']}


## Section D — сравнение методов → Таблица 1 (пропустится, если посчитано)

In [ ]:
# ============================================================
# SECTION D — сравнение методов (Таблица 1). RESUME-aware.
# ============================================================
tl, vl = make_loaders(train_idx, val_idx)
R = load_R()
for mth in COMPARE_METHODS:
    if mth in R["compare"]:
        print(f"skip {mth} (готово)"); continue
    print(f"\n===== {NAMES_RU[mth]} =====")
    m = train_one(mth, tl, vl, epochs=CFG.epochs, tag=NAMES_RU[mth])
    R["compare"][mth] = clean(m); save_R(R)
    print(f"  -> acc={m['accuracy']:.3f} auc={m['roc_auc']:.3f} f1={m['f1']:.3f} H={m['hidden_entropy_bits']:.2f}")

rows=[]
for mth in COMPARE_METHODS:
    if mth not in R["compare"]: continue
    m=R["compare"][mth]
    rows.append({"Метод":NAMES_RU[mth],"Acc %":round(m["accuracy"]*100,1),
        "Bal.Acc %":round(m["bal_accuracy"]*100,1),"Sens":round(m["recall_sens"],3),
        "Spec":round(m["specificity"],3),"F1":round(m["f1"],3),
        "ROC-AUC":round(m["roc_auc"],3),"PR-AUC":round(m["pr_auc"],3),
        "H, бит":round(m["hidden_entropy_bits"],2)})
table1=pd.DataFrame(rows); print("\n===== ТАБЛИЦА 1 ====="); print(table1.to_string(index=False))
table1.to_csv(f"{RESULTS_DIR}/table1_comparison.csv", index=False)


skip baseline (готово)
skip l2 (готово)
skip dropout (готово)
skip l1_act (готово)
skip vib (готово)
skip aes (готово)

===== ТАБЛИЦА 1 =====
              Метод  Acc %  Bal.Acc %  Sens  Spec    F1  ROC-AUC  PR-AUC  H, бит
       Базовая (CE)   66.7       66.7 0.600 0.733 0.643    0.738   0.784   10.17
  L2 (weight decay)   70.0       70.0 0.600 0.800 0.667    0.762   0.810   10.36
            Dropout   63.3       63.3 0.733 0.533 0.667    0.762   0.833    9.71
       L1 активаций   66.7       66.7 0.400 0.933 0.545    0.760   0.809   10.97
VIB (стохастич. IB)   70.0       70.0 0.600 0.800 0.667    0.724   0.822   11.47
 AES (предложенный)   70.0       70.0 0.733 0.667 0.710    0.711   0.790    0.27


## Section E — ablation AES → Таблица 2 (пропустится, если посчитано)

In [ ]:
# ============================================================
# SECTION E — ablation AES (Таблица 2). RESUME-aware (ключ = имя конфигурации).
# ============================================================
ABLATION = ([(f"gamma={g}", {"gamma":g}) for g in [0.10,0.25,0.50,0.75]] +
            [(f"alpha={a}", {"alpha":a}) for a in [0.00,0.01,0.05,0.10]] +
            [(f"deep={L}", {"deep_layer":L}) for L in ["layer3","layer4"]] +
            [(f"warmup={w}", {"warmup":w}) for w in [0,4]])
R = load_R()
for name, ov in ABLATION:
    if name in R["ablation"]:
        print(f"skip {name} (готово)"); continue
    print(f"\n----- {name} -----")
    m = train_one("aes", tl, vl, epochs=CFG.epochs_ablation, overrides=ov, verbose=False, tag=f"AES|{name}")
    R["ablation"][name] = clean(m); save_R(R)
    print(f"  acc={m['accuracy']:.3f} auc={m['roc_auc']:.3f} f1={m['f1']:.3f} H={m['hidden_entropy_bits']:.2f}")

rows=[]
for name,_ in ABLATION:
    if name not in R["ablation"]: continue
    m=R["ablation"][name]
    rows.append({"Конфигурация":name,"Acc %":round(m["accuracy"]*100,1),"F1":round(m["f1"],3),
        "ROC-AUC":round(m["roc_auc"],3),"PR-AUC":round(m["pr_auc"],3),"H, бит":round(m["hidden_entropy_bits"],2)})
table2=pd.DataFrame(rows); print("\n===== ТАБЛИЦА 2 (ablation) ====="); print(table2.to_string(index=False))
table2.to_csv(f"{RESULTS_DIR}/table2_ablation.csv", index=False)


skip gamma=0.1 (готово)
skip gamma=0.25 (готово)
skip gamma=0.5 (готово)
skip gamma=0.75 (готово)
skip alpha=0.0 (готово)
skip alpha=0.01 (готово)
skip alpha=0.05 (готово)
skip alpha=0.1 (готово)
skip deep=layer3 (готово)
skip deep=layer4 (готово)
skip warmup=0 (готово)
skip warmup=4 (готово)

===== ТАБЛИЦА 2 (ablation) =====
Конфигурация  Acc %    F1  ROC-AUC  PR-AUC  H, бит
   gamma=0.1   70.0 0.690    0.711   0.755    0.12
  gamma=0.25   70.0 0.710    0.711   0.790    0.27
   gamma=0.5   70.0 0.690    0.729   0.790    0.33
  gamma=0.75   73.3 0.692    0.709   0.770    0.67
   alpha=0.0   66.7 0.643    0.738   0.784   10.17
  alpha=0.01   70.0 0.667    0.640   0.705    0.34
  alpha=0.05   70.0 0.710    0.711   0.790    0.27
   alpha=0.1   70.0 0.710    0.724   0.776    0.19
 deep=layer3   70.0 0.727    0.756   0.790    0.34
 deep=layer4   70.0 0.710    0.711   0.790    0.27
    warmup=0   70.0 0.640    0.751   0.792    0.25
    warmup=4   70.0 0.710    0.711   0.790    0.27


## Section F — k-fold (layer4) → Таблица 3 (пропустится, если посчитано)

In [ ]:
# ============================================================
# SECTION F — k-fold CV (Таблица 3). RESUME-aware (ключ = "foldI:method").
# ============================================================
CV_METHODS = ["baseline","vib","aes"]
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
R = load_R()
for fold,(tr_i,va_i) in enumerate(skf.split(all_idx, labels)):
    tlf, vlf = make_loaders(tr_i, va_i)
    for mth in CV_METHODS:
        key=f"fold{fold+1}:{mth}"
        if key in R["cv"]:
            print(f"skip {key}"); continue
        m = train_one(mth, tlf, vlf, epochs=CFG.epochs_cv, verbose=False, tag=key)
        R["cv"][key]=clean(m); save_R(R)
        print(f"{key}: acc={m['accuracy']:.3f} auc={m['roc_auc']:.3f} f1={m['f1']:.3f} H={m['hidden_entropy_bits']:.2f}")

def agg(metric):
    out={}
    for mth in CV_METHODS:
        vals=[R["cv"][f"fold{f+1}:{mth}"][metric] for f in range(CFG.n_folds)
              if f"fold{f+1}:{mth}" in R["cv"]]
        v=np.array(vals,dtype=float); out[mth]=f"{np.nanmean(v):.3f} ± {np.nanstd(v):.3f}"
    return out
acc,auc,f1,Hh = agg("accuracy"),agg("roc_auc"),agg("f1"),agg("hidden_entropy_bits")
table3=pd.DataFrame([{"Метод":NAMES_RU[m],"Acc":acc[m],"ROC-AUC":auc[m],"F1":f1[m],"H, бит":Hh[m]} for m in CV_METHODS])
print(f"\n===== ТАБЛИЦА 3 ({CFG.n_folds}-fold, mean ± std) ====="); print(table3.to_string(index=False))
table3.to_csv(f"{RESULTS_DIR}/table3_cv.csv", index=False)


skip fold1:baseline
skip fold1:vib
skip fold1:aes
skip fold2:baseline
skip fold2:vib
skip fold2:aes
skip fold3:baseline
skip fold3:vib
skip fold3:aes
skip fold4:baseline
skip fold4:vib
skip fold4:aes
skip fold5:baseline
skip fold5:vib
skip fold5:aes

===== ТАБЛИЦА 3 (5-fold, mean ± std) =====
              Метод           Acc       ROC-AUC            F1        H, бит
       Базовая (CE) 0.660 ± 0.053 0.701 ± 0.056 0.655 ± 0.028 6.255 ± 2.161
VIB (стохастич. IB) 0.673 ± 0.053 0.743 ± 0.068 0.630 ± 0.085 9.194 ± 1.496
 AES (предложенный) 0.653 ± 0.027 0.665 ± 0.062 0.564 ± 0.100 0.212 ± 0.068


## Section F2 — **НОВОЕ**: k-fold для AES на layer3 → Таблица 3'
Считается только AES@layer3; baseline/VIB/AES@layer4 берутся из прошлого прогона на тех же фолдах.

In [ ]:
# ============================================================
# SECTION F2 — CV для AES на layer3 (один заранее выбранный кандидат из ablation)
#   baseline / VIB / AES@layer4 берутся из уже посчитанного results.json (те же фолды).
#   Новое считается только для AES@layer3 -> ключ "foldI:aes_l3".
# ============================================================
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
R = load_R()
for fold,(tr_i,va_i) in enumerate(skf.split(all_idx, labels)):
    key = f"fold{fold+1}:aes_l3"
    if key in R["cv"]:
        print(f"skip {key}"); continue
    tlf, vlf = make_loaders(tr_i, va_i)
    m = train_one("aes", tlf, vlf, epochs=CFG.epochs_cv,
                  overrides={"deep_layer":"layer3"}, verbose=False, tag=key)
    R["cv"][key] = clean(m); save_R(R)
    print(f"{key}: acc={m['accuracy']:.3f} auc={m['roc_auc']:.3f} f1={m['f1']:.3f} H={m['hidden_entropy_bits']:.2f}")

def agg2(method_key, metric):
    vals=[R["cv"][f"fold{f+1}:{method_key}"][metric] for f in range(CFG.n_folds)
          if f"fold{f+1}:{method_key}" in R["cv"]]
    v=np.array(vals,dtype=float); return f"{np.nanmean(v):.3f} ± {np.nanstd(v):.3f}"

ROWS = [("baseline","Базовая (CE)"), ("vib","VIB (стохастич. IB)"),
        ("aes","AES @ layer4"), ("aes_l3","AES @ layer3")]
tableF2 = pd.DataFrame([{"Метод":name,
    "Acc":agg2(k,"accuracy"), "ROC-AUC":agg2(k,"roc_auc"),
    "F1":agg2(k,"f1"), "H, бит":agg2(k,"hidden_entropy_bits")}
    for k,name in ROWS if f"fold1:{k}" in R["cv"]])
print(f"\n===== ТАБЛИЦА 3' ({CFG.n_folds}-fold, mean ± std): layer3 vs layer4 =====")
print(tableF2.to_string(index=False))
tableF2.to_csv(f"{RESULTS_DIR}/table3b_cv_layer3.csv", index=False)


fold1:aes_l3: acc=0.700 auc=0.750 f1=0.769 H=0.42
fold2:aes_l3: acc=0.633 auc=0.585 f1=0.732 H=0.60
fold3:aes_l3: acc=0.733 auc=0.676 f1=0.733 H=0.59
fold4:aes_l3: acc=0.633 auc=0.707 f1=0.522 H=0.32
fold5:aes_l3: acc=0.700 auc=0.676 f1=0.609 H=0.30

===== ТАБЛИЦА 3' (5-fold, mean ± std): layer3 vs layer4 =====
              Метод           Acc       ROC-AUC            F1        H, бит
       Базовая (CE) 0.660 ± 0.053 0.701 ± 0.056 0.655 ± 0.028 6.255 ± 2.161
VIB (стохастич. IB) 0.673 ± 0.053 0.743 ± 0.068 0.630 ± 0.085 9.194 ± 1.496
       AES @ layer4 0.653 ± 0.027 0.665 ± 0.062 0.564 ± 0.100 0.212 ± 0.068
       AES @ layer3 0.680 ± 0.040 0.679 ± 0.054 0.673 ± 0.093 0.445 ± 0.127
